# Patrón Estructural: Bridge

## Introducción
El patrón Bridge permite dividir una clase grande o un grupo de clases estrechamente relacionadas en dos jerarquías separadas (abstracción e implementación) que pueden desarrollarse independientemente.

## Objetivos
- Comprender cómo separar la abstracción de la implementación.
- Identificar cuándo es útil el patrón Bridge.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: App de Banco**
Supón que tienes diferentes tipos de cuentas (Ahorros, Corriente) y diferentes sistemas de notificación (Email, SMS). El patrón Bridge permite combinar cualquier tipo de cuenta con cualquier sistema de notificación sin crear una clase para cada combinación.

**¿Dónde se usa en proyectos reales?**
En sistemas de pagos, aplicaciones bancarias, frameworks de UI, etc.

## Sin patrón Bridge (forma errónea)
Se crean clases para cada combinación posible, lo que genera mucho código duplicado.

In [1]:
class CuentaAhorrosEmail:
    def notificar(self, mensaje):
        print(f'Email: {mensaje}')

class CuentaAhorrosSMS:
    def notificar(self, mensaje):
        print(f'SMS: {mensaje}')

# Si agregas más tipos, el número de clases crece exponencialmente

## Con patrón Bridge (forma correcta)
Separas la abstracción (Cuenta) de la implementación (Notificador).

In [2]:
class Notificador:
    def notificar(self, mensaje):
        pass

class Email(Notificador):
    def notificar(self, mensaje):
        print(f'Email: {mensaje}')

class SMS(Notificador):
    def notificar(self, mensaje):
        print(f'SMS: {mensaje}')

class Cuenta:
    def __init__(self, notificador):
        self.notificador = notificador
    def enviar_alerta(self, mensaje):
        self.notificador.notificar(mensaje)

cuenta = Cuenta(Email())
cuenta.enviar_alerta('Saldo bajo')

Email: Saldo bajo


## UML del patrón Bridge
```plantuml
@startuml
class Cuenta {
    + enviar_alerta(mensaje)
}
class Notificador {
    + notificar(mensaje)
}
Cuenta --> Notificador
Notificador <|-- Email
Notificador <|-- SMS
@enduml
```

## Otro ejemplo de la vida real: Generación de reportes en múltiples formatos
**Contexto:** un sistema de BI genera distintos tipos de informe (ventas, inventario) que pueden exportarse en distintos formatos (PDF, Excel). El *tipo* de informe y el *formato* de salida son dos dimensiones que varían de forma independiente — justo lo que Bridge está hecho para resolver.

### Sin patrón (forma errónea)
Una clase por cada combinación tipo × formato. Con solo 2 tipos y 2 formatos ya son 4 clases; agregar un tercer formato (CSV) obligaría a crear 2 clases más.

In [3]:
class InformeVentasPDF:
    def generar(self):
        print('Generando PDF:\nVentas totales: $12.500.000')

class InformeVentasExcel:
    def generar(self):
        print('Generando Excel:\nVentas totales: $12.500.000')

class InformeInventarioPDF:
    def generar(self):
        print('Generando PDF:\nInventario: 340 unidades disponibles')

class InformeInventarioExcel:
    def generar(self):
        print('Generando Excel:\nInventario: 340 unidades disponibles')

# Si agregas más tipos de informe o más formatos, el número de clases crece exponencialmente

### Con patrón (forma correcta)
Se separa la abstracción (`Informe`, con sus subtipos `InformeVentas`/`InformeInventario`) de la implementación (`Exportador`, con `ExportadorPDF`/`ExportadorExcel`). Cualquier tipo de informe puede combinarse con cualquier formato sin crear una clase nueva.

In [4]:
class Exportador:
    def exportar(self, contenido):
        raise NotImplementedError

class ExportadorPDF(Exportador):
    def exportar(self, contenido):
        print(f'Generando PDF:\n{contenido}')

class ExportadorExcel(Exportador):
    def exportar(self, contenido):
        print(f'Generando Excel:\n{contenido}')


class Informe:
    def __init__(self, exportador):
        self.exportador = exportador
    def generar(self):
        raise NotImplementedError


class InformeVentas(Informe):
    def generar(self):
        contenido = 'Ventas totales: $12.500.000'
        self.exportador.exportar(contenido)


class InformeInventario(Informe):
    def generar(self):
        contenido = 'Inventario: 340 unidades disponibles'
        self.exportador.exportar(contenido)


informe = InformeVentas(ExportadorPDF())
informe.generar()

informe2 = InformeInventario(ExportadorExcel())
informe2.generar()

Generando PDF:
Ventas totales: $12.500.000
Generando Excel:
Inventario: 340 unidades disponibles


### UML del ejemplo de reportes
```plantuml
@startuml
abstract class Informe {
    + generar()
}
class InformeVentas
class InformeInventario
Informe <|-- InformeVentas
Informe <|-- InformeInventario
Informe --> Exportador

abstract class Exportador {
    + exportar(contenido)
}
class ExportadorPDF
class ExportadorExcel
Exportador <|-- ExportadorPDF
Exportador <|-- ExportadorExcel
@enduml
```

### ¿Dónde más se usa Bridge?
- **Reportes/BI:** exactamente este ejemplo — separar el tipo de reporte del formato de salida (PDF, Excel, CSV).
- **Frameworks de UI multiplataforma:** separar el control visual (Botón, Ventana) de su renderizado nativo (Windows, macOS, Linux), como hacen Qt o Java AWT/Swing.
- **Drivers gráficos:** separar la forma geométrica (Círculo, Rectángulo) de la API de dibujo usada (OpenGL, DirectX, Canvas web).
- **Notificaciones multicanal:** separar el tipo de evento (Alerta, Recordatorio) del canal de envío (Email, SMS, Push), evitando una clase por cada combinación.
- **Persistencia poliglota:** separar la entidad de dominio (Usuario, Pedido) del motor de almacenamiento (PostgreSQL, MongoDB) sin una clase `UsuarioPostgres`, `UsuarioMongo`, etc.

**Ejercicio de reflexión:** agrega un tercer formato, `ExportadorCSV`. ¿Cuántas clases nuevas tuviste que crear con Bridge? ¿Cuántas habrías necesitado en la versión "sin patrón" para cubrir los mismos 2 tipos de informe?

## Actividad
Crea tu propio Bridge para combinar diferentes tipos de cuentas y sistemas de autenticación (por ejemplo, biometría, token, contraseña).

---
## Explicación de conceptos clave
- **Separación de abstracción e implementación:** Permite desarrollar ambas jerarquías de forma independiente.
- **Escalabilidad:** Facilita la extensión sin crear una explosión de clases.
- **Aplicación en la vida real:** Útil en sistemas con múltiples combinaciones de funcionalidades.

## Conclusión
El patrón Bridge es ideal para evitar la explosión de clases cuando tienes múltiples dimensiones de variación. Es común en aplicaciones bancarias, sistemas de notificación y frameworks de UI.